# 03 — Run Evaluation
Runs RAGAS across every (chunking × retrieval) combination and saves a comparison table.
Make sure 02_build_index.ipynb has been run for each chunking strategy first.

In [23]:
# ── Environment & path setup ─────────────────────────────────────────────────
import os, sys
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd().parent  # notebooks/ -> project root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
QDRANT_URL     = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")

print("OPENAI_API_KEY set:", bool(OPENAI_API_KEY))
print("QDRANT_URL       :", QDRANT_URL)

OPENAI_API_KEY set: True
QDRANT_URL       : https://38718a4e-130d-4ef7-be90-0c1893444e4c.us-east4-0.gcp.cloud.qdrant.io


In [24]:
# ── Clients ──────────────────────────────────────────────────────────────────
import torch
from openai import OpenAI
from qdrant_client import QdrantClient
from fastembed import SparseTextEmbedding
from sentence_transformers import CrossEncoder
import config

oai           = OpenAI(api_key=OPENAI_API_KEY)
qdrant        = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY or None)
bm25_model    = SparseTextEmbedding(model_name="Qdrant/bm25")
cross_encoder = CrossEncoder(
    config.RERANK_MODEL,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
print("All clients ready.")

All clients ready.


In [25]:
# ── Generate GT pairs (only needed once — skipped if file exists) ─────────────
import os, json
from eval.generate_gt import generate_gt_pairs, load_gt_pairs

if os.path.exists(config.GT_PAIRS_PATH):
    print('GT pairs already exist — loading.')
    gt_pairs = load_gt_pairs()
else:
    # Generate from the recursive collection (chunking doesn't affect GT quality much)
    gt_pairs = generate_gt_pairs(
        oai=oai,
        qdrant=qdrant,
        collection_name='hr_rag_recursive',
    )

print(f'{len(gt_pairs)} GT pairs ready.')

GT pairs already exist — loading.
50 GT pairs ready.


In [ ]:
# ── Run evaluation grid ────────────────────────────────────────────────────────
import importlib
import eval.evaluate as _evaluate_mod
importlib.reload(_evaluate_mod)
from eval.evaluate import run_eval_grid, print_summary

df = run_eval_grid(
    gt_pairs=gt_pairs,
    oai=oai,
    qdrant=qdrant,
    bm25_model=bm25_model,
    cross_encoder=cross_encoder,
    openai_api_key=OPENAI_API_KEY,
    # Optionally limit scope during development:
    # chunking_strategies=['recursive', 'semantic', 'sentence_window'],
    # retrieval_strategies=['dense', 'hybrid', 'compression'],
    chunking_strategies=['semantic', 'sentence_window'],
    retrieval_strategies=['compression'],
    include_rerank=True,  # True: also runs dense/hybrid with cross-encoder rerank; False: run subset scope only
)

print_summary(df)

In [ ]:
# single eval, manual append results
from eval.evaluate import run_eval_one

row = run_eval_one(
    chunking="semantic",
    retrieval="compression",
    gt_pairs=gt_pairs,
    oai=oai,
    qdrant=qdrant,
    bm25_model=bm25_model,
    cross_encoder=cross_encoder,
    openai_api_key=OPENAI_API_KEY,
    use_rerank=True,   # <-- reranking on
)
print(row)

In [37]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent
df = pd.read_csv(ROOT / "results" / "eval_results.csv")
print_summary(df)


EVALUATION SUMMARY
                           config  faithfulness  answer_relevancy  context_precision  context_recall
       recursive / dense + rerank        0.7837            0.7419             0.8591          0.9450
      recursive / hybrid + rerank        0.7879            0.7698             0.8682          0.9167
               recursive / hybrid        0.7390            0.6457             0.8440          0.8900
                recursive / dense        0.7627            0.7447             0.8257          0.8707
        semantic / dense + rerank        0.7583            0.6650             0.7692          0.8467
                semantic / hybrid        0.7465            0.7129             0.8067          0.8433
       semantic / hybrid + rerank        0.7555            0.6878             0.7849          0.8367
                 semantic / dense        0.6822            0.6562             0.7478          0.8267
         sentence_window / hybrid        0.4939            0.6341      

In [39]:
# ── Pivot table: GT context_recall by chunking × retrieval ─────────────────────
import pandas as pd

pivot = df[~df['rerank']].pivot_table(
    index='chunking', columns='retrieval', values='context_recall'
).round(4)

print('GT context recall (no rerank):')
print(pivot)

# Best overall config (by GT context recall)
best = df.loc[df['context_recall'].idxmax()]
print(f"\nBest config: {best['config']}")
print(f"  context_recall : {best['context_recall']:.4f}")
print(f"  faithfulness   : {best['faithfulness']:.4f}")
print(f"  answer_relevancy: {best['answer_relevancy']:.4f}")

GT context recall (no rerank):
retrieval         dense  hybrid
chunking                       
recursive        0.8707  0.8900
semantic         0.8267  0.8433
sentence_window  0.5517  0.5783

Best config: recursive / dense + rerank
  context_recall : 0.9450
  faithfulness   : 0.7837
  answer_relevancy: 0.7419
